In [ ]:
# Imports and Initial Setup
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from datasets.RawVessels.loader import RawVesselsDataset

import numpy as np
import cv2
# Additional imports for statistical threshold calculation
from scipy.optimize import minimize
from matplotlib import pyplot as plt
from scipy.ndimage import gaussian_filter, median_filter
from scipy.signal import hilbert
from skimage import io
from skimage.restoration import denoise_tv_chambolle
from skimage.feature import blob_log, peak_local_max

# ----- Load the dataset ------
root_dir_datamodule = '/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/data/TASI/DataSAR_real_refined'
# Ensure the root directory exists
if not Path(root_dir_datamodule).exists():
    raise FileNotFoundError(f"Root directory {root_dir_datamodule} does not exist.")
# Load image and mask paths
root_dir_datamodule = Path(root_dir_datamodule)
if not (root_dir_datamodule / 'inputs').exists() or not (root_dir_datamodule / 'masks').exists():
    raise FileNotFoundError(f"Expected directories 'inputs' and 'masks' not found in {root_dir_datamodule}.")

image_paths = [x for x in (root_dir_datamodule / 'inputs').glob('*.pkl')]
mask_paths = [x for x in (root_dir_datamodule / 'masks').glob('*.pkl')]




In [ ]:
from scipy.ndimage import label, center_of_mass


loader = RawVesselsDataset(image_paths, 
                           mask_paths, 
                           transform=None)


# ------ DataLoader Setup ------
# Try loading and inspecting a sample
rand_idx = 1000  # Change this index to load different samples
sample = loader[rand_idx]
img, mask = sample 








# ----- Image Processing ------
Re, Im = img  # Real and Imaginary parts of the complex image
# Make Amplitude and Phase
amp = np.abs(Re + 1j * Im)  # Amplitude
phase = np.angle(Re + 1j * Im)  # Phase

# Compute mean and std for phase
mean = np.mean(phase)
std = np.std(phase)
vmin = mean - 0.5 * std
vmax = mean + 0.5 * std

fig, axs = plt.subplots(1, 3, figsize=(14, 7), dpi=140)

axs[0].imshow(amp, cmap='gray')
axs[0].set_title('Amplitude')
axs[0].axis('off')


# ----- Centoid Calculation ------
# Label connected components in the mask
labeled_mask, num_features = label(mask)

# Calculate the center of mass for each labeled instance
centroids = center_of_mass(mask, labeled_mask, range(1, num_features + 1))
n, S = 4, 201
print(f"Number of instances: {num_features}")
print(f"Centroids: {centroids}")


# Increased contrast for phase
# Apply Gaussian smoothing to the phase before displaying
phase_smoothed = gaussian_filter(phase, sigma=1.0)
axs[1].imshow(phase_smoothed, cmap='inferno', vmin=vmin, vmax=vmax)
axs[1].set_title('Phase (Smoothed, Increased Contrast)')

for centroid in centroids:
    center_x, center_y = int(centroid[1]), int(centroid[0])  # Convert to integer coordinates
    theta = np.linspace(0, 2 * np.pi, n, endpoint=False)
    for angle in theta:
        x_end = center_x + S * np.cos(angle)
        y_end = center_y + S * np.sin(angle)
        axs[1].plot([center_x, x_end], [center_y, y_end], color='cyan', linewidth=1)

    axs[1].scatter(center_x, center_y, color='red', s=10, label='Center')
axs[1].legend(loc='upper right', fontsize='small')

# Overlay radial sampling information on axs[2]
axs[2].imshow(mask, cmap='gray')
axs[2].set_title('Ground Truth Mask')
axs[2].axis('off')

for centroid in centroids:
    center_x, center_y = int(centroid[1]), int(centroid[0])  # Convert to integer coordinates
    for angle in theta:
        x_end = center_x + S * np.cos(angle)
        y_end = center_y + S * np.sin(angle)
        axs[2].plot([center_x, x_end], [center_y, y_end], color='cyan', linewidth=1)

    axs[2].scatter(center_x, center_y, color='red', s=10, label='Center')

# Ensure the legend is added only once
axs[2].legend(loc='upper right', fontsize='small')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.ndimage import gaussian_filter1d
from radial_sampler import radial_sampling


# ----- Config ------

idx_centroid = 0  # Assuming we want to sample from the first centroid
sigma = 2.0  # Standard deviation for Gaussian smoothing
random = True  # Set to True if you want to sample randomly from the centroids



# ----- Radial Sampling ------
center_x, center_y = int(centroids[idx_centroid][1]), int(centroids[idx_centroid][0])  # Use the first centroid for sampling

samples = radial_sampling(phase, center_x, center_y, S, n)
# Apply Gaussian smoothing to each sample
smoothed_samples = [gaussian_filter1d(sample, sigma=sigma) for sample in samples]







# ----- Plot ------
# Plot the original and smoothed samples
fig, axs = plt.subplots(4, 3, figsize=(14, 12), dpi=140)

# Flatten the axs array for easier indexing
axs = axs.flatten()

for i, smoothed_sample in enumerate(smoothed_samples):
    ax = axs[i]  # Use flattened indexing for a 4x3 grid
    ax.plot(smoothed_sample, label="Smoothed")
    ax.plot(samples[i], alpha=0.5, label="Original")  # Overlay original for comparison
    ax.set_title(f"Sample {i+1}")
    ax.grid(True)
    ax.legend()

plt.tight_layout()
plt.show()



# Find the minimum length among all smoothed samples
min_length = min(sample.shape[0] for sample in smoothed_samples)

# Truncate all smoothed samples to the minimum length
truncated_samples = [sample[:min_length] for sample in smoothed_samples]

# Calculate the common up or down trend between truncated samples
common_trend = np.mean(truncated_samples, axis=0)

fig, ax = plt.subplots(figsize=(10, 6), dpi=140)

# Plot the common trend
ax.plot(common_trend, label="Common Trend", color="blue", linewidth=2)

# Highlight regions where the trend is positive or negative
positive_trend = np.where(common_trend > 0, common_trend, np.nan)
negative_trend = np.where(common_trend < 0, common_trend, np.nan)

ax.fill_between(range(len(common_trend)), positive_trend, 0, color="green", alpha=0.3, label="Positive Trend")
ax.fill_between(range(len(common_trend)), negative_trend, 0, color="red", alpha=0.3, label="Negative Trend")

ax.set_title("Common Up or Down Trend Between Samples")
ax.set_xlabel("Index")
ax.set_ylabel("Value")
ax.grid(True)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import ruptures as rpt
from scipy.stats import norm
from scipy.special import logsumexp
import warnings
warnings.filterwarnings('ignore')


# === Bayesian Changepoint Detection ===

class BayesianChangepointDetection:
    """
    Simple Bayesian Changepoint Detection implementation
    Based on the approach used in BCP and similar to BEAST methodology
    """
    
    def __init__(self, prior_prob=0.01, variance_scale=1.0):
        self.prior_prob = prior_prob  # Prior probability of changepoint
        self.variance_scale = variance_scale
    
    def detect_changepoints(self, data, min_size=3):
        """
        Detect changepoints using Bayesian approach
        Returns changepoint locations and their probabilities
        """
        n = len(data)
        # Log probabilities for dynamic programming
        log_prob = np.full((n, n), -np.inf)
        
        # Precompute segment statistics
        for i in range(n):
            for j in range(i + min_size - 1, n):
                segment = data[i:j+1]
                if len(segment) >= min_size:
                    # Log likelihood of segment with constant mean
                    mean_seg = np.mean(segment)
                    var_seg = np.var(segment) * self.variance_scale
                    if var_seg > 0:
                        log_prob[i, j] = np.sum(norm.logpdf(segment, mean_seg, np.sqrt(var_seg)))
        
        # Dynamic programming to find optimal segmentation
        dp = np.full(n + 1, -np.inf)
        dp[0] = 0
        parent = np.zeros(n + 1, dtype=int)
        
        for t in range(1, n + 1):
            for s in range(max(0, t - n), t):
                if t - s >= min_size and log_prob[s, t-1] != -np.inf:
                    # Cost includes segment likelihood and changepoint penalty
                    cost = dp[s] + log_prob[s, t-1] + np.log(self.prior_prob)
                    if cost > dp[t]:
                        dp[t] = cost
                        parent[t] = s
        
        # Backtrack to find changepoints
        changepoints = []
        t = n
        while t > 0:
            prev_t = parent[t]
            if prev_t > 0:
                changepoints.append(prev_t)
            t = prev_t
        
        return sorted(changepoints)

# === Multiple Changepoint Detection Methods ===

def detect_changepoints_ruptures(data, method='rbf', n_bkps=5):
    """
    Use ruptures library for changepoint detection
    """
    # Try different algorithms
    algorithms = {
        'rbf': rpt.KernelCPD(kernel="rbf"),
        'linear': rpt.KernelCPD(kernel="linear"),
        'pelt': rpt.Pelt(model="rbf"),
        'binseg': rpt.Binseg(model="l2"),
        'window': rpt.Window(width=40, model="l2")
    }
    
    results = {}
    
    for name, algo in algorithms.items():
        try:
            if name in ['pelt']:
                # PELT automatically determines number of changepoints
                changepoints = algo.fit(data.reshape(-1, 1)).predict(pen=10)
            else:
                # Other methods need explicit number of changepoints
                changepoints = algo.fit(data.reshape(-1, 1)).predict(n_bkps=min(n_bkps, len(data)//10))
            
            # Remove the last point (end of series) if present
            if changepoints[-1] == len(data):
                changepoints = changepoints[:-1]
            
            results[name] = changepoints
        except Exception as e:
            print(f"Algorithm {name} failed: {e}")
            results[name] = []
    
    return results

def classify_extrema_from_changepoints(data, changepoints):
    """
    Classify changepoints as peaks or valleys based on local behavior
    """
    peaks = []
    valleys = []
    
    # Add boundaries
    all_points = sorted([0] + list(changepoints) + [len(data) - 1])
    
    for i, point in enumerate(all_points):
        if point == 0 or point == len(data) - 1:
            continue
            
        # Look at local neighborhood
        window = 5
        start_idx = max(0, point - window)
        end_idx = min(len(data), point + window + 1)
        local_data = data[start_idx:end_idx]
        local_point = point - start_idx
        
        if local_point < len(local_data):
            # Check if it's a local maximum or minimum
            left_mean = np.mean(local_data[:local_point]) if local_point > 0 else data[point]
            right_mean = np.mean(local_data[local_point+1:]) if local_point < len(local_data)-1 else data[point]
            center_val = data[point]
            
            if center_val > left_mean and center_val > right_mean:
                peaks.append(point)
            elif center_val < left_mean and center_val < right_mean:
                valleys.append(point)
    
    return np.array(peaks), np.array(valleys)

# === Apply Bayesian Changepoint Detection ===

print("=== Bayesian Changepoint Detection Results ===")
print(f"Data length: {len(common_trend)}")
print(f"Data range: [{np.min(common_trend):.3f}, {np.max(common_trend):.3f}]")

# Method 1: Custom Bayesian implementation
print("\n1. Custom Bayesian Changepoint Detection:")
bcp_detector = BayesianChangepointDetection(prior_prob=0.05, variance_scale=1.0)
changepoints_bcp = bcp_detector.detect_changepoints(common_trend)
peaks_bcp, valleys_bcp = classify_extrema_from_changepoints(common_trend, changepoints_bcp)

print(f"   Changepoints: {changepoints_bcp}")
print(f"   Detected peaks: {peaks_bcp}")
print(f"   Detected valleys: {valleys_bcp}")
print(f"   Peak values: {[f'{common_trend[p]:.3f}' for p in peaks_bcp]}")
print(f"   Valley values: {[f'{common_trend[v]:.3f}' for v in valleys_bcp]}")

# Method 2: Ruptures library (multiple algorithms)
print("\n2. Ruptures Library Methods:")
changepoints_ruptures = detect_changepoints_ruptures(common_trend, n_bkps=6)

for method, cps in changepoints_ruptures.items():
    if len(cps) > 0:
        peaks_rupt, valleys_rupt = classify_extrema_from_changepoints(common_trend, cps)
        print(f"   {method.upper()}: {len(cps)} changepoints, {len(peaks_rupt)} peaks, {len(valleys_rupt)} valleys")
        print(f"      Changepoints: {cps}")
        if len(peaks_rupt) > 0:
            print(f"      Peaks at: {peaks_rupt} (values: {[f'{common_trend[p]:.3f}' for p in peaks_rupt]})")
        if len(valleys_rupt) > 0:
            print(f"      Valleys at: {valleys_rupt} (values: {[f'{common_trend[v]:.3f}' for v in valleys_rupt]})")

# Compare with original scipy method
from scipy.signal import find_peaks
peaks_scipy, _ = find_peaks(common_trend)
valleys_scipy, _ = find_peaks(-common_trend)

print("\n3. Original Scipy Method (for comparison):")
print(f"   Scipy peaks: {peaks_scipy} (values: {[f'{common_trend[p]:.3f}' for p in peaks_scipy]})")
print(f"   Scipy valleys: {valleys_scipy} (values: {[f'{common_trend[v]:.3f}' for v in valleys_scipy]})")



In [ ]:
# === Visualization of Bayesian Changepoint Detection ===

# Ensure common_trend is defined and not empty before plotting
if 'common_trend' in locals() and common_trend.size > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=140)
    axes = axes.flatten()

    # Plot 1: Custom Bayesian Changepoint Detection
    ax1 = axes[0]
    ax1.plot(common_trend, 'b-', linewidth=2, label='Common Trend', alpha=0.7)
    if peaks_bcp.size > 0:
        ax1.scatter(peaks_bcp, common_trend[peaks_bcp.astype(int)], color='red', s=80, marker='^', 
                   label=f'BCP Peaks ({len(peaks_bcp)})', zorder=5)
    if valleys_bcp.size > 0:
        ax1.scatter(valleys_bcp, common_trend[valleys_bcp.astype(int)], color='green', s=80, marker='v', 
                   label=f'BCP Valleys ({len(valleys_bcp)})', zorder=5)
    # Mark changepoints
    for cp in changepoints_bcp:
        ax1.axvline(x=cp, color='orange', linestyle='--', alpha=0.6)
    ax1.set_title('Custom Bayesian Changepoint Detection')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    ax1.set_xlabel('Index')
    ax1.set_ylabel('Value')

    # Plot 2: Best Ruptures method (choose PELT or RBF)
    best_method = 'pelt' if 'pelt' in changepoints_ruptures and changepoints_ruptures['pelt'].size > 0 else 'rbf'
    if best_method in changepoints_ruptures and changepoints_ruptures[best_method].size > 0:
        cps_best = changepoints_ruptures[best_method]
        peaks_best, valleys_best = classify_extrema_from_changepoints(common_trend, cps_best)
        
        ax2 = axes[1]
        ax2.plot(common_trend, 'b-', linewidth=2, label='Common Trend', alpha=0.7)
        if peaks_best.size > 0:
            ax2.scatter(peaks_best, common_trend[peaks_best.astype(int)], color='red', s=80, marker='^', 
                       label=f'{best_method.upper()} Peaks ({len(peaks_best)})', zorder=5)
        if valleys_best.size > 0:
            ax2.scatter(valleys_best, common_trend[valleys_best.astype(int)], color='green', s=80, marker='v', 
                       label=f'{best_method.upper()} Valleys ({len(valleys_best)})', zorder=5)
        # Mark changepoints
        for cp in cps_best:
            ax2.axvline(x=cp, color='purple', linestyle='--', alpha=0.6)
        ax2.set_title(f'Ruptures {best_method.upper()} Method')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        ax2.set_xlabel('Index')
        ax2.set_ylabel('Value')
    else:
        axes[1].text(0.5, 0.5, 'No valid ruptures results for selected method', transform=axes[1].transAxes, 
                    ha='center', va='center', fontsize=12)
        axes[1].set_title('Ruptures Method (No Data)')

    # Plot 3: Original Scipy method
    ax3 = axes[2]
    ax3.plot(common_trend, 'b-', linewidth=2, label='Common Trend', alpha=0.7)
    if peaks_scipy.size > 0:
        ax3.scatter(peaks_scipy, common_trend[peaks_scipy.astype(int)], color='red', s=80, marker='^', 
                   label=f'Scipy Peaks ({len(peaks_scipy)})', zorder=5)
    if valleys_scipy.size > 0:
        ax3.scatter(valleys_scipy, common_trend[valleys_scipy.astype(int)], color='green', s=80, marker='v', 
                   label=f'Scipy Valleys ({len(valleys_scipy)})', zorder=5)
    ax3.set_title('Original Scipy find_peaks Method')
    ax3.grid(True, alpha=0.3)
    ax3.legend()
    ax3.set_xlabel('Index')
    ax3.set_ylabel('Value')

    # Plot 4: Comparison of all methods
    ax4 = axes[3]
    ax4.plot(common_trend, 'b-', linewidth=2, label='Common Trend', alpha=0.7)

    # Add all detected peaks/valleys with different markers
    if peaks_bcp.size > 0:
        ax4.scatter(peaks_bcp, common_trend[peaks_bcp.astype(int)], color='red', s=60, marker='^', 
                   label=f'BCP Peaks', alpha=0.8, edgecolors='darkred')
    if valleys_bcp.size > 0:
        ax4.scatter(valleys_bcp, common_trend[valleys_bcp.astype(int)], color='green', s=60, marker='v', 
                   label=f'BCP Valleys', alpha=0.8, edgecolors='darkgreen')

    if best_method in changepoints_ruptures and changepoints_ruptures[best_method].size > 0:
        # peaks_best, valleys_best would have been defined above if this condition is met
        if peaks_best.size > 0:
            ax4.scatter(peaks_best, common_trend[peaks_best.astype(int)], color='orange', s=40, marker='s', 
                       label=f'{best_method.upper()} Peaks', alpha=0.7)
        if valleys_best.size > 0:
            ax4.scatter(valleys_best, common_trend[valleys_best.astype(int)], color='cyan', s=40, marker='s', 
                       label=f'{best_method.upper()} Valleys', alpha=0.7)

    if peaks_scipy.size > 0:
        ax4.scatter(peaks_scipy, common_trend[peaks_scipy.astype(int)], color='red', s=20, marker='o', 
                   label=f'Scipy Peaks', alpha=0.6)
    if valleys_scipy.size > 0:
        ax4.scatter(valleys_scipy, common_trend[valleys_scipy.astype(int)], color='green', s=20, marker='o', 
                   label=f'Scipy Valleys', alpha=0.6)

    ax4.set_title('Comparison of All Methods')
    ax4.grid(True, alpha=0.3)
    ax4.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax4.set_xlabel('Index')
    ax4.set_ylabel('Value')

    plt.tight_layout()
    plt.show()

    # === Statistical Analysis ===
    print("\\n=== Statistical Analysis ===")
    print(f"Total data points: {len(common_trend)}")
    print(f"Data standard deviation: {np.std(common_trend):.4f}")
    print(f"Data mean: {np.mean(common_trend):.4f}")

    # Compare methods
    methods_summary = {
        'Custom BCP': {'peaks': len(peaks_bcp), 'valleys': len(valleys_bcp)},
        'Scipy': {'peaks': len(peaks_scipy), 'valleys': len(valleys_scipy)}
    }

    if best_method in changepoints_ruptures and changepoints_ruptures[best_method].size > 0 :
        # peaks_best, valleys_best defined if condition met
        methods_summary[f'Ruptures {best_method.upper()}'] = {
            'peaks': len(peaks_best), 'valleys': len(valleys_best)
        }

    print("\\nMethod comparison:")
    for method_name, stats in methods_summary.items():
        total_extrema = stats['peaks'] + stats['valleys']
        print(f"{method_name:20}: {stats['peaks']:2d} peaks, {stats['valleys']:2d} valleys, {total_extrema:2d} total")

    # Calculate metrics for alternating pattern
    if peaks_bcp.size > 0 and valleys_bcp.size > 0: # Check if arrays are not empty
        extremes_bcp = np.sort(np.concatenate((peaks_bcp, valleys_bcp)))
        if extremes_bcp.size > 1: # Need at least 2 points to check alternation
            is_alternating_bcp = all(((extremes_bcp[i] in peaks_bcp) != (extremes_bcp[i-1] in peaks_bcp))
                                    for i in range(1, len(extremes_bcp)))
            print(f"\\nBCP alternating pattern: {is_alternating_bcp}")
        
            # Calculate amplitude consistency
            diffs_bcp = [abs(common_trend[extremes_bcp[i].astype(int)] - common_trend[extremes_bcp[i-1].astype(int)])
                         for i in range(1, len(extremes_bcp))]
            print(f"BCP amplitude differences: {[f'{d:.3f}' for d in diffs_bcp]}")
            print(f"BCP amplitude std: {np.std(diffs_bcp):.4f}")
else:
    print("Skipping visualization and statistical analysis as 'common_trend' is not defined or is empty.")

In [ ]:
# === Advanced BEAST-like Bayesian Changepoint Detection ===
# This implements a more sophisticated approach similar to BEAST (Bayesian Estimator of Abrupt change, Seasonality, and Trend)

from scipy.stats import invgamma, multivariate_normal, norm # Already imported norm, logsumexp
from collections import defaultdict

class BEASTLikeDetector:
    """
    Advanced Bayesian changepoint detection inspired by BEAST methodology
    Uses reversible-jump MCMC for model selection and parameter estimation
    """
    
    def __init__(self, max_changepoints=10, burnin=1000, n_samples=5000):
        self.max_changepoints = max_changepoints
        self.burnin = burnin
        self.n_samples = n_samples
        
    def log_likelihood(self, data, changepoints, means, variance):
        """Calculate log likelihood of data given segmentation"""
        if variance <= 0: # Guard against invalid variance
            return -np.inf
        if len(changepoints) == 0:
            if len(means) == 0: return -np.inf # Should not happen if estimate_parameters is correct
            return np.sum(norm.logpdf(data, means[0], np.sqrt(variance)))
        
        ll = 0
        segments = [0] + list(changepoints) + [len(data)]
        
        for i in range(len(segments) - 1):
            start, end = segments[i], segments[i + 1]
            segment_data = data[start:end]
            if len(segment_data) > 0:
                if i >= len(means): return -np.inf # Should not happen
                ll += np.sum(norm.logpdf(segment_data, means[i], np.sqrt(variance)))
        
        return ll
    
    def propose_changepoint_move(self, changepoints, data_length, move_type='add'):
        """Propose adding, removing, or moving a changepoint"""
        new_changepoints = list(changepoints) # Work with a list copy
        
        min_segment_len = 5 # Minimum length of a segment

        if move_type == 'add' and len(new_changepoints) < self.max_changepoints:
            # Add a new changepoint ensuring segments are not too small
            possible_cps = [cp for cp in range(min_segment_len, data_length - min_segment_len) if cp not in new_changepoints]
            # Further filter to ensure new_cp maintains min_segment_len for all segments
            valid_new_cps = []
            for cp_candidate in possible_cps:
                temp_cps = sorted(new_changepoints + [cp_candidate])
                all_segment_lengths = np.diff([0] + temp_cps + [data_length])
                if np.all(all_segment_lengths >= min_segment_len):
                    valid_new_cps.append(cp_candidate)
            
            if valid_new_cps:
                new_cp = np.random.choice(valid_new_cps)
                new_changepoints.append(new_cp)
                new_changepoints.sort()
        
        elif move_type == 'remove' and len(new_changepoints) > 0:
            idx_to_remove = np.random.randint(len(new_changepoints))
            # Check if removing maintains min_segment_len
            temp_cps = new_changepoints[:idx_to_remove] + new_changepoints[idx_to_remove+1:]
            all_segment_lengths = np.diff([0] + temp_cps + [data_length])
            if np.all(all_segment_lengths >= min_segment_len) or not temp_cps : # Allow removal if it results in 0 cps
                 new_changepoints.pop(idx_to_remove)
        
        elif move_type == 'move' and len(new_changepoints) > 0:
            idx_to_move = np.random.randint(len(new_changepoints))
            old_cp = new_changepoints[idx_to_move]
            
            # Define bounds for the move to maintain order and min_segment_len
            lower_bound = new_changepoints[idx_to_move - 1] + min_segment_len if idx_to_move > 0 else min_segment_len
            upper_bound = new_changepoints[idx_to_move + 1] - min_segment_len if idx_to_move < len(new_changepoints) - 1 else data_length - min_segment_len
            
            # Propose a new position within a small window around old_cp, respecting bounds
            move_range = 10
            proposed_pos = old_cp + np.random.randint(-move_range, move_range + 1)
            new_cp = max(lower_bound, min(upper_bound, proposed_pos))

            if new_cp != old_cp and new_cp not in new_changepoints: # Ensure it actually moved and is valid
                # Check segment lengths with the new_cp
                temp_list = new_changepoints[:]
                temp_list[idx_to_move] = new_cp
                temp_list.sort() # Important if new_cp jumped over another cp
                all_segment_lengths = np.diff([0] + temp_list + [data_length])
                if np.all(all_segment_lengths >= min_segment_len):
                    new_changepoints[idx_to_move] = new_cp
                    new_changepoints.sort() # Ensure sorted order
        
        return new_changepoints
    
    def estimate_parameters(self, data, changepoints):
        """Estimate segment means and global variance"""
        segments = [0] + list(changepoints) + [len(data)]
        means = []
        all_residuals = []
        
        for i in range(len(segments) - 1):
            start, end = segments[i], segments[i + 1]
            segment_data = data[start:end]
            if len(segment_data) > 0:
                mean_est = np.mean(segment_data)
                means.append(mean_est)
                all_residuals.extend(segment_data - mean_est)
            else:
                # This case should be avoided by min_segment_len in propose_changepoint_move
                means.append(np.mean(data) if len(data)>0 else 0) # Fallback for empty segment
        
        if not means: # If there are no segments (e.g. data is empty, or changepoints make all segments empty)
             means.append(np.mean(data) if len(data)>0 else 0)


        if len(all_residuals) > 1:
            variance = np.var(all_residuals)
        elif len(data)>0 : # If only one segment or few points
            variance = np.var(data) if np.var(data) > 0 else 1.0
        else: # No data
            variance = 1.0
            
        return means, max(variance, 1e-9)  # Avoid zero or too small variance
    
    def detect_changepoints_mcmc(self, data):
        """Run MCMC to detect changepoints"""
        if len(data) < 10 : # Need enough data for MCMC
            print("Data too short for BEAST-like MCMC detection.")
            return [], defaultdict(int), 0.0

        current_changepoints = [] # Start with no changepoints
        changepoint_counts = defaultdict(int)
        accepted_proposals = 0
        total_proposals = 0
        
        # MCMC sampling
        for iteration in range(self.burnin + self.n_samples):
            move_types = ['add', 'remove', 'move']
            # Adjust probs: if no cps, must add. if max_cps, cannot add.
            if not current_changepoints:
                move_probs = [1.0, 0.0, 0.0]
            elif len(current_changepoints) >= self.max_changepoints:
                move_probs = [0.0, 0.5, 0.5]
            else:
                move_probs = [0.4, 0.3, 0.3]
            move_type = np.random.choice(move_types, p=move_probs)
            
            proposed_changepoints = self.propose_changepoint_move(
                current_changepoints, len(data), move_type
            )
            
            current_means, current_var = self.estimate_parameters(data, current_changepoints)
            proposed_means, proposed_var = self.estimate_parameters(data, proposed_changepoints)
            
            current_ll = self.log_likelihood(data, current_changepoints, current_means, current_var)
            proposed_ll = self.log_likelihood(data, proposed_changepoints, proposed_means, proposed_var)
            
            # Prior for number of changepoints (e.g., Poisson prior)
            lambda_cps = 1 # Expected number of changepoints
            current_prior_cps = -lambda_cps + len(current_changepoints) * np.log(lambda_cps) # Log Poisson PMF (ignoring factorial)
            proposed_prior_cps = -lambda_cps + len(proposed_changepoints) * np.log(lambda_cps)

            # Jacobian for dimension-changing moves (add/remove) - simplified here
            log_jacobian = 0
            if move_type == 'add' and proposed_changepoints != current_changepoints: # if add was successful
                log_jacobian = np.log(len(data)) # Simplified, relates to proposal density
            elif move_type == 'remove' and proposed_changepoints != current_changepoints: # if remove was successful
                log_jacobian = -np.log(len(data))

            log_alpha = (proposed_ll + proposed_prior_cps) - (current_ll + current_prior_cps) + log_jacobian
            
            total_proposals += 1
            if np.log(np.random.random()) < log_alpha: # Compare in log space
                current_changepoints = proposed_changepoints
                accepted_proposals += 1
            
            if iteration >= self.burnin:
                for cp in current_changepoints:
                    changepoint_counts[cp] += 1
        
        threshold_prob = 0.2 # Changepoint must appear in at least 20% of post-burnin samples
        significant_changepoints = [
            cp for cp, count in changepoint_counts.items() 
            if (count / self.n_samples) >= threshold_prob
        ]
        
        acceptance_rate = accepted_proposals / total_proposals if total_proposals > 0 else 0.0
        
        return sorted(significant_changepoints), changepoint_counts, acceptance_rate

# === Apply BEAST-like Detection ===
# Ensure common_trend is defined and not empty
if 'common_trend' in locals() and common_trend.size > 0:
    print("\\n=== BEAST-like Bayesian Changepoint Detection ===")
    beast_detector = BEASTLikeDetector(max_changepoints=min(8, len(common_trend)//10), burnin=1000, n_samples=3000) # Adjusted params

    print("Running MCMC sampling... (this may take a moment)")
    changepoints_beast, posterior_counts, acceptance_rate = beast_detector.detect_changepoints_mcmc(common_trend)

    print(f"MCMC acceptance rate: {acceptance_rate:.3f}")
    print(f"BEAST-like changepoints: {changepoints_beast}")

    peaks_beast, valleys_beast = classify_extrema_from_changepoints(common_trend, changepoints_beast)

    print(f"BEAST peaks: {peaks_beast} (values: {[f'{common_trend[p.astype(int)]:.3f}' for p in peaks_beast if p < len(common_trend)]})")
    print(f"BEAST valleys: {valleys_beast} (values: {[f'{common_trend[v.astype(int)]:.3f}' for v in valleys_beast if v < len(common_trend)]})")

    print("\\nPosterior changepoint probabilities:")
    total_samples_mcmc = beast_detector.n_samples
    if total_samples_mcmc > 0:
        for cp_val in sorted(posterior_counts.keys()):
            prob = posterior_counts[cp_val] / total_samples_mcmc
            if prob >= 0.05: 
                print(f"   Position {cp_val}: {prob:.3f} ({posterior_counts[cp_val]}/{total_samples_mcmc} samples)")

    # === Enhanced Visualization ===
    fig, axes_beast = plt.subplots(2, 2, figsize=(16, 10), dpi=140) # Renamed axes to avoid clash

    # Plot 1: BEAST results with posterior probabilities
    ax1_b = axes_beast[0, 0]
    ax1_b.plot(common_trend, 'b-', linewidth=2, label='Common Trend', alpha=0.7)

    if total_samples_mcmc > 0:
        for cp_val, count in posterior_counts.items():
            prob = count / total_samples_mcmc
            if prob >= 0.05:
                alpha_val = min(1.0, prob * 2) 
                ax1_b.axvline(x=cp_val, color='red', linestyle='--', alpha=alpha_val, 
                           linewidth=2 if prob > 0.3 else 1)
    if peaks_beast.size > 0:
        ax1_b.scatter(peaks_beast, common_trend[peaks_beast.astype(int)], color='red', s=100, marker='^', 
                   label=f'BEAST Peaks ({len(peaks_beast)})', zorder=5, edgecolors='darkred')
    if valleys_beast.size > 0:
        ax1_b.scatter(valleys_beast, common_trend[valleys_beast.astype(int)], color='green', s=100, marker='v', 
                   label=f'BEAST Valleys ({len(valleys_beast)})', zorder=5, edgecolors='darkgreen')

    ax1_b.set_title('BEAST-like Detection (with Posterior Probabilities)')
    ax1_b.grid(True, alpha=0.3)
    ax1_b.legend()
    ax1_b.set_xlabel('Index')
    ax1_b.set_ylabel('Value')

    # Plot 2: Posterior probability bar plot
    ax2_b = axes_beast[0, 1]
    posterior_probs_values = np.zeros(len(common_trend))
    if total_samples_mcmc > 0:
        for cp_val, count in posterior_counts.items():
            if 0 <= cp_val < len(posterior_probs_values):
                posterior_probs_values[cp_val] = count / total_samples_mcmc
    
    ax2_b.plot(common_trend, 'b-', linewidth=2, alpha=0.3, label='Common Trend') # Make trend less prominent
    ax2_b.set_xlabel('Index')
    ax2_b.set_ylabel('Common Trend Value', color='blue')
    ax2_b.tick_params(axis='y', labelcolor='blue')

    ax2_b_twin = ax2_b.twinx()
    ax2_b_twin.bar(range(len(posterior_probs_values)), posterior_probs_values, 
                     alpha=0.6, color='red', width=1.0, label='Posterior Prob.')
    ax2_b_twin.set_ylabel('Posterior Probability', color='red')
    ax2_b_twin.tick_params(axis='y', labelcolor='red')
    ax2_b.set_title('Changepoint Posterior Probabilities')
    # ax2_b.grid(True, alpha=0.3) # Grid might be too busy with twin axes
    lines, labels = ax2_b.get_legend_handles_labels()
    lines2, labels2 = ax2_b_twin.get_legend_handles_labels()
    ax2_b_twin.legend(lines + lines2, labels + labels2, loc='upper right')


    # Plot 3: Method comparison (BEAST, BCP, Scipy)
    ax3_b = axes_beast[1, 0]
    ax3_b.plot(common_trend, 'b-', linewidth=2, label='Common Trend', alpha=0.7)

    methods_results_plot = {
        'BEAST': (peaks_beast, valleys_beast, 'red', 'green'),
        'BCP': (peaks_bcp if 'peaks_bcp' in locals() else np.array([]), valleys_bcp if 'valleys_bcp' in locals() else np.array([]), 'orange', 'cyan'),
        'Scipy': (peaks_scipy if 'peaks_scipy' in locals() else np.array([]), valleys_scipy if 'valleys_scipy' in locals() else np.array([]), 'purple', 'yellow')
    }

    for i, (method_name_plot, (pks, vls, pk_col, vl_col)) in enumerate(methods_results_plot.items()):
        offset = i * 0.01 * np.std(common_trend) # Slight offset based on data scale
        if pks.size > 0:
            ax3_b.scatter(pks, common_trend[pks.astype(int)] + offset, 
                       color=pk_col, s=60, marker='^', alpha=0.8,
                       label=f'{method_name_plot} Peaks ({len(pks)})')
        if vls.size > 0:
            ax3_b.scatter(vls, common_trend[vls.astype(int)] - offset, 
                       color=vl_col, s=60, marker='v', alpha=0.8,
                       label=f'{method_name_plot} Valleys ({len(vls)})')

    ax3_b.set_title('Comparison of Detection Methods')
    ax3_b.grid(True, alpha=0.3)
    ax3_b.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
    ax3_b.set_xlabel('Index')
    ax3_b.set_ylabel('Value')

    # Plot 4: Segmentation visualization for BEAST
    ax4_b = axes_beast[1, 1]
    ax4_b.plot(common_trend, 'b-', linewidth=2, alpha=0.7, label='Data')

    beast_segments = [0] + sorted(changepoints_beast) + [len(common_trend)]
    # Ensure segments are unique and ordered for diff
    beast_segments = sorted(list(set(beast_segments))) 
    
    num_unique_segments = len(beast_segments) -1
    if num_unique_segments > 0:
        segment_colors = plt.cm.viridis(np.linspace(0, 1, num_unique_segments))

        for i in range(num_unique_segments):
            start, end = beast_segments[i], beast_segments[i + 1]
            if start >= end: continue # Skip empty or invalid segments
            segment_data_plot = common_trend[start:end]
            if segment_data_plot.size > 0:
                segment_mean_plot = np.mean(segment_data_plot)
                
                ax4_b.axhline(y=segment_mean_plot, xmin=start/len(common_trend), xmax=end/len(common_trend),
                           color=segment_colors[i], linewidth=3, alpha=0.7)
                ax4_b.axvspan(start, end, alpha=0.1, color=segment_colors[i])
                
                mid_point_plot = (start + end) // 2
                ax4_b.text(mid_point_plot, segment_mean_plot, f'S{i+1}', 
                        ha='center', va='bottom', fontweight='bold', fontsize=8, color=segment_colors[i]*0.8) # Darker text

    for cp_val in changepoints_beast:
        ax4_b.axvline(x=cp_val, color='red', linestyle='--', alpha=0.8)

    ax4_b.set_title('BEAST Segmentation')
    ax4_b.grid(True, alpha=0.3)
    ax4_b.set_xlabel('Index')
    ax4_b.set_ylabel('Value')
    ax4_b.legend()

    plt.tight_layout()
    plt.show()

    # === Final Comparison ===
    print("\\n=== Final Method Comparison ===")
    final_methods_comp = {
        'BEAST-like MCMC': {'peaks': len(peaks_beast), 'valleys': len(valleys_beast), 'total_cp': len(changepoints_beast)},
        'Custom BCP': {'peaks': len(peaks_bcp) if 'peaks_bcp' in locals() else 0, 'valleys': len(valleys_bcp) if 'valleys_bcp' in locals() else 0, 'total_cp': len(changepoints_bcp) if 'changepoints_bcp' in locals() else 0},
        'Scipy find_peaks': {'peaks': len(peaks_scipy) if 'peaks_scipy' in locals() else 0, 'valleys': len(valleys_scipy) if 'valleys_scipy' in locals() else 0, 'total_cp': (len(peaks_scipy) if 'peaks_scipy' in locals() else 0) + (len(valleys_scipy) if 'valleys_scipy' in locals() else 0)}
    }

    print(f"{'Method':<20} {'Peaks':<6} {'Valleys':<8} {'Total CP':<8} {'Quality Metrics'}")
    print("-" * 70)

    for method_name_comp, stats_comp in final_methods_comp.items():
        quality = "N/A"
        if method_name_comp == 'BEAST-like MCMC' and peaks_beast.size > 0 and valleys_beast.size > 0:
            extremes = np.sort(np.concatenate((peaks_beast, valleys_beast)))
            if extremes.size > 1:
                alternating = all(((extremes[i] in peaks_beast) != (extremes[i-1] in peaks_beast))
                                 for i in range(1, len(extremes)))
                quality = f"Alt: {alternating}, Acc: {acceptance_rate:.2f}"
        elif method_name_comp == 'Custom BCP' and ('peaks_bcp' in locals() and peaks_bcp.size > 0) and ('valleys_bcp' in locals() and valleys_bcp.size > 0):
            extremes = np.sort(np.concatenate((peaks_bcp, valleys_bcp)))
            if extremes.size > 1:
                alternating = all(((extremes[i] in peaks_bcp) != (extremes[i-1] in peaks_bcp))
                                 for i in range(1, len(extremes)))
                quality = f"Alt: {alternating}"
        
        print(f"{method_name_comp:<20} {stats_comp['peaks']:<6} {stats_comp['valleys']:<8} {stats_comp['total_cp']:<8} {quality}")

    print("\\nRecommendation: The BEAST-like MCMC method can provide robust")
    print("changepoint detection with uncertainty quantification. Review acceptance rates and posterior probabilities.")
else:
    print("Skipping BEAST-like MCMC detection and visualization as 'common_trend' is not defined or is empty.")